# ODI to Databricks Migration

## Source File: `TARGET`
**Conversion Timestamp:** 2024-07-30T12:00:00Z
**Description:** Load data into `TRG_LOC` from `LOCATIONS`.

In [ ]:
import datetime

dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type (FULL/INC)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

# Derive current extract time from current_timestamp for consistency if not passed
current_extract_time_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.000000")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", current_extract_time_str, "Current Extract Timestamp")

# Placeholder for last extract time, typically fetched from a control table
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00.000000", "Last Extract Timestamp")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id,
  CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid,
  CAST(${ODI_SESS_NO} AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS') AS etl_current_extract_time,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss.SSSSSS') AS etl_last_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Target Table Setup

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
-- Create target table if it does not exist (inferred DDL)
CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (
    LOCATION_ID     BIGINT,
    STREET_ADDRESS  STRING,
    POSTAL_CODE     STRING,
    CITY            STRING,
    STATE_PROVINCE  STRING,
    COUNTRY_ID      STRING
) USING DELTA;

## Load into Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
-- Insert data into target table
INSERT INTO workspace.hr.trg_loc
(
    LOCATION_ID,
    STREET_ADDRESS,
    POSTAL_CODE,
    CITY,
    STATE_PROVINCE,
    COUNTRY_ID
)
SELECT
    LOCATIONS.LOCATION_ID,
    LOCATIONS.STREET_ADDRESS,
    LOCATIONS.POSTAL_CODE,
    LOCATIONS.CITY,
    LOCATIONS.STATE_PROVINCE,
    LOCATIONS.COUNTRY_ID
FROM
    workspace.hr.locations AS LOCATIONS;

## Cleanup (No temporary tables in this flow)

## Validation

In [ ]:
%sql
SELECT 'TRG_LOC' AS table_name, COUNT(*) AS record_count FROM workspace.hr.trg_loc;

## Conversion Notes and Manual Actions Required

1.  **Inferred DDL:** The `CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc` statement was inferred based on common Oracle data types and column names. Please verify these data types (`BIGINT`, `STRING`) against the actual source Oracle DDL for `HR.TRG_LOC` and `HR.LOCATIONS` to ensure correctness.
2.  **No MERGE Operation:** The original ODI script only contained a direct `INSERT` statement. If the `TRG_LOC` table is intended to be a SCD Type 1 or Type 2 table requiring updates/inserts based on keys, this script will need to be converted to a `MERGE INTO` statement. As provided, it performs a full reload or append.
3.  **Source Table Existence:** This notebook assumes that `workspace.hr.locations` table already exists and contains the necessary data.